# Notebook 04 — Revenue Prediction Model

Builds and evaluates game-day total revenue prediction models.
Target: `total_game_day_revenue = net_ticket_revenue + concessions + merchandise`

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from src.config import DATA_PROCESSED_DIR, MODELS_DIR
sns.set_theme(style='whitegrid')
print('Setup complete')

## 1. Revenue Target Construction

In [ ]:
df = pd.read_csv(DATA_PROCESSED_DIR / 'features_revenue.csv', parse_dates=['game_date'])
print(f'Revenue feature dataset: {len(df):,} rows')
print('\nRevenue target stats:')
print(df['total_game_day_revenue'].describe().round(0))

plt.figure(figsize=(10, 4))
plt.hist(df['total_game_day_revenue'], bins=50, color='steelblue', edgecolor='white')
plt.title('Distribution of Total Game-Day Revenue')
plt.xlabel('Total Revenue ($)')
plt.tight_layout()
plt.show()

## 2. Train Revenue Models

In [ ]:
from src.train_revenue_model import main as train_revenue
train_revenue()

## 3. Revenue Prediction Visualisation

In [ ]:
preds = pd.read_csv(DATA_PROCESSED_DIR / 'revenue_predictions.csv')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter(preds['total_game_day_revenue'], preds['predicted_total_revenue'], alpha=0.4, color='darkorange', s=15)
max_val = max(preds['total_game_day_revenue'].max(), preds['predicted_total_revenue'].max())
axes[0].plot([0, max_val], [0, max_val], 'r--', lw=1.5)
axes[0].set_xlabel('Actual Revenue')
axes[0].set_ylabel('Predicted Revenue')
axes[0].set_title('Actual vs Predicted Game-Day Revenue')

errors = preds['revenue_prediction_error']
axes[1].hist(errors, bins=40, color='coral', edgecolor='white')
axes[1].axvline(0, color='navy', linestyle='--')
axes[1].set_title('Revenue Prediction Error Distribution')
axes[1].set_xlabel('Error ($)')

plt.tight_layout()
plt.show()

mae  = mean_absolute_error(preds['total_game_day_revenue'], preds['predicted_total_revenue'])
rmse = np.sqrt(mean_squared_error(preds['total_game_day_revenue'], preds['predicted_total_revenue']))
r2   = r2_score(preds['total_game_day_revenue'], preds['predicted_total_revenue'])
print(f'MAE: ${mae:,.0f} | RMSE: ${rmse:,.0f} | R²: {r2:.4f}')

## 4. Feature Importances & Business Interpretation

In [ ]:
import json
with open(MODELS_DIR / 'model_features.json') as f:
    meta = json.load(f)

fi = meta.get('revenue_feature_importances', {})
if fi:
    fi_df = pd.Series(fi).sort_values(ascending=False).head(12)
    fi_df.plot(kind='barh', figsize=(10, 5), color='darkorange')
    plt.title('Top Feature Importances — Revenue Model')
    plt.xlabel('Importance')
    plt.gca().invert_yaxis()
    plt.tight_layout()
    plt.show()
    print('\nBusiness interpretation:')
    print('Actual attendance is the dominant revenue driver — it multiplies across tickets, food, and merch.')
    print('Ticket price averages and opponent popularity are secondary predictors.')
    print('Models can be used to pre-estimate revenue for upcoming games to guide staffing and ordering.')